# Explainable Research Paper Retrieval and Recommendation System
## Semantic Search • Summarization • Keyword Extraction • Explainable AI • Difficulty Analysis

In [1]:
# Install all required external libraries
!pip install datasets transformers==4.46.3 keybert==0.8.5 faiss-cpu gliner

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 682.4 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of gliner to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 41.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizer

## Dataset Loading

In [2]:
from datasets import load_dataset

In [3]:
# Load the Machine Learning ArXiv Papers dataset from Hugging Face
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

In [4]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [5]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

## Data Preprocessing

In [6]:
import pandas as pd

In [7]:
# Convert Hugging Face dataset into a Pandas DataFrame and display first few rows
df = pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [8]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [9]:
df = df[['title','abstract']]

In [10]:
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [11]:
df.shape

(117592, 2)

In [12]:
# Select only the first 15,000 research papers.
# This reduces computation time while generating embeddings.
df = df.head(15000)
df.shape

(15000, 2)

In [13]:
df.isnull().sum()

,0
title,0
abstract,0


In [14]:
# Combine the title and abstract into a single column.This combined text will be used for generating embeddings.
df["paper_text"] = df["title"] + " " + df["abstract"]

/tmp/ipykernel_1065/3633062628.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"] = df["title"] + " " + df["abstract"]


In [15]:
df["paper_text"].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [16]:
# Display the paper_text column as a DataFrame
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [17]:
type(df[["paper_text"]])

pandas.core.frame.DataFrame

In [18]:
print(df['paper_text'].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

## Embedding Generation

**Load Sentence Transformer**

In [19]:
# Import SentenceTransformer
# It converts text into dense numerical vectors (embeddings).
from sentence_transformers import SentenceTransformer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [20]:
# Load the pre-trained MiniLM embedding model
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [22]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex= False)

df["paper_text"] = df["paper_text"].str.strip()

/tmp/ipykernel_1065/4163540211.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex= False)
/tmp/ipykernel_1065/4163540211.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"] = df["paper_text"].str.strip()


In [23]:
# Select the first research paper for testing the embedding model.
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

**Generate Embedding**

In [24]:
# Convert the sample paper into a numerical embedding vector.
embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [25]:
# Display the first 56 values of the embedding vector
embedding[:56]

array([-0.13156408, -0.00678264, -0.00367611,  0.03265159,  0.11219642,
        0.01227268,  0.09816723, -0.09005231,  0.0423116 , -0.01977348,
       -0.03308417,  0.07452946,  0.10632041, -0.0206043 , -0.02052102,
        0.00169494,  0.0708195 ,  0.05854455, -0.11231909,  0.02082473,
        0.05692543,  0.0201578 ,  0.02583109,  0.0321703 ,  0.10513762,
       -0.09676764,  0.02700803, -0.02345091, -0.04549675, -0.010137  ,
       -0.01794854, -0.04814429,  0.01077654, -0.03759069,  0.01943482,
        0.0371519 ,  0.02967845,  0.04330943,  0.04373214,  0.03704866,
       -0.00182595,  0.00455184, -0.00799067,  0.03037367, -0.014378  ,
        0.03795146,  0.05959162, -0.02583359, -0.06521574,  0.05900269,
       -0.02107136,  0.07359424, -0.05720104,  0.00294058,  0.00767514,
       -0.03334163], dtype=float32)

**Generate Embeddings for First 5 Papers**

In [26]:
# Generate embeddings for the first 5 research papers.
sample_embedding = model.encode(
    df["paper_text"].head(5).to_list()
)

# Check the shape of the generated embeddings
sample_embedding.shape

(5, 384)

**Check Similarity of Same Paper**

In [27]:
# Import cosine similarity function
# Cosine similarity measures how similar two vectors are.

from sklearn.metrics.pairwise import cosine_similarity

# Compare the first paper with itself. Similarity should be close to 1.0.
similarity = cosine_similarity(
    sample_embedding[0].reshape(1, -1),
    sample_embedding[0].reshape(1, -1)
)
print(similarity)

[[1.]]


**Compare Two Different Papers**

In [28]:
# Compare the first paper with the second paper.
# Since they are different research papers, the similarity score will be less than 1.

similarity = cosine_similarity(
    sample_embedding[0].reshape(1, -1),
    sample_embedding[1].reshape(1, -1)
)
print(similarity)

[[0.36625266]]


In [29]:
# Compare the first paper with the remaining four papers.
# This helps verify that semantically similar papers receive higher similarity scores.

for i in range(1, 5):
    sim = cosine_similarity(
        sample_embedding[0].reshape(1, -1),
        sample_embedding[i].reshape(1, -1)
    )

    print(f"Paper 0 vs Paper {i}")
    print(sim)

Paper 0 vs Paper 1
[[0.36625266]]
Paper 0 vs Paper 2
[[0.33522838]]
Paper 0 vs Paper 3
[[0.1550511]]
Paper 0 vs Paper 4
[[0.37421542]]


**Generate full embedding on 15000 research papers**

In [31]:
# Generate embeddings for all research papers.
# Batch processing is used to improve speed and memory efficiency.

embedding = model.encode(
    df["paper_text"].to_list(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [32]:
print(embedding.shape)
print(type(embedding))

(15000, 384)
<class 'numpy.ndarray'>


**Check Embedding Data Type**

In [33]:
embedding.dtype

dtype('float32')

**Import FAISS**

In [34]:
import faiss

**Normalize Embeddings**

In [35]:
# Normalize all embedding vectors using L2 normalization.
# This allows Inner Product (IP) similarity to behave like Cosine Similarity.
faiss.normalize_L2(embedding)

## FAISS Index Creation

In [36]:
# IndexFlatIP performs exact similarity search using Inner Product.
index = faiss.IndexFlatIP(384)

In [37]:
# Add all embedding vectors to the FAISS index.
index.add(embedding)

**Generate and Save Embeddings**

In [39]:
# Import libraries for saving embeddings and checking whether files already exist.
# Check whether the embeddings file already exists. If it exists, load the saved embeddings to save computation time.
# Otherwise, generate embeddings and save them for future use.
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    # Save embeddings as a NumPy file
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

Generating embeddings


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Embeddings saved successfully!


**Save or Load FAISS Index**

In [40]:
# Check whether the FAISS index already exists.
# If it exists, load it directly.Otherwise, create a new FAISS index, add embeddings, and save it for future use.

if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")

else:
    print("Creating new FAISS index")

    # Normalize embeddings before adding them to the FAISS index
    faiss.normalize_L2(embeddings)

    # Create FAISS Index using Inner Product similarity
    index = faiss.IndexFlatIP(384)

    # Add embeddings into the index
    index.add(embeddings)

    # Save the FAISS index
    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")

Creating new FAISS index
FAISS index saved successfully!


In [41]:
# Display the total number of research papers stored inside the FAISS index.
print(index.ntotal)

15000


**User Query**

In [42]:
# Enter the search query.
# The recommendation system will retrieve semantically similar research papers.
query = "deep learning for medical image analysis"

**Generate Query Embedding**

In [43]:
# Convert the user query into an embedding vector.
query_embedding = model.encode([query])

# Display embedding shape
query_embedding.shape

(1, 384)

**Normalize Query Embedding**

In [44]:
# Normalize the query embedding.
# This ensures that Inner Product similarity behaves like Cosine Similarity.
faiss.normalize_L2(query_embedding)

**Search Similar Papers**

In [45]:
# Search the FAISS index.

# D = Similarity Scores
# I = Indices of retrieved research papers
# k = Number of top similar papers to retrieve

D, I = index.search(query_embedding, 5)

print(D)
print(I)

[[0.6807244  0.67092204 0.65219986 0.6281174  0.6131152 ]]
[[10466 13730 11873 12691 11282]]


**Display Retrieved Paper Title**

In [46]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


**Display Retrieved Paper Abstract**

In [47]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



**Display Another Retrieved Paper**

In [48]:
print(df.iloc[11873]["title"])

Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection


In [49]:
# Observation:
# Sometimes the retrieved papers may not contain the exact keywords used in the query.
# This is expected because Sentence Transformers perform Semantic Search instead of Keyword Search.
# Semantic Search retrieves papers based on meaning, not exact word matching.

## Semantic Search

In [50]:
# Create a reusable function to search the top-k similar research papers.
def search_paper(query, k=5):

    # Convert query into embedding
    query_embedding = model.encode([query])

    # Normalize the query embedding
    faiss.normalize_L2(query_embedding)

    # Search the FAISS index
    D, I = index.search(query_embedding, k)

    return D, I

**Test Search Function**

In [51]:
D,I = search_paper("deep learning for medical image analysis")
print(D)
print(I)

[[0.6807244  0.67092204 0.65219986 0.6281174  0.6131152 ]]
[[10466 13730 11873 12691 11282]]


**Display Search Results**

In [52]:
# Update the search function to display the similarity score,paper title, and a portion of the abstract.
def search_paper(query,k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(embeddings)
  D,I = index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score",score)
    print("Paper Title",df.iloc[idx]["title"])
    print("Paper Abstract",df.iloc[idx]["abstract"][:500])
    print()

**Search Research Papers**

In [53]:
# Retrieve and display the top matching research papers.
search_paper("deep learning for medical image analysis")

Similarity Score 0.6807244
Paper Title A Perspective on Deep Imaging
Paper Abstract   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity Score 0.67092204
Paper Title Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Paper Abstract   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-traine

## Research Paper Summarization

**Load the BART Summarization Model**

In [55]:
# Import the pipeline function from Transformers.
from transformers import pipeline

# Load Facebook's BART Large CNN model for abstractive text summarization.
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

type(summarizer)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

transformers.pipelines.text2text_generation.SummarizationPipeline

**Generate Summary for One Paper**

In [56]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [57]:
type(summary)

list

In [58]:
type(summary[0])

dict

In [59]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

**Summarize All Retrieved Papers**

In [60]:
# Display similarity score, paper title, abstract, and generated summary for each retrieved paper.

for score, idx in zip(D[0], I[0]):
    print("Similarity Score :", score)
    print("Paper Title :", df.iloc[idx]["title"])
    print("Paper Abstract :")
    print(df.iloc[idx]["abstract"][:500])

    # Generate summary
    summary = summarizer(
        df.iloc[idx]["abstract"],
        max_length=120,
        min_length=40
    )

    print("Paper Summary :")
    print(summary[0]["summary_text"])
    print()

Similarity Score : 0.6807244
Paper Title : A Perspective on Deep Imaging
Paper Abstract :
  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
Paper Summary :
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity Score : 0.67092204
Paper Title : Convolu

**Search and Summarize Research Papers**

In [64]:
def search_paper_and_summarize(query,k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(embeddings)
  D,I = index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score :",score)
    print("Paper Title :",df.iloc[idx]["title"])
    print("Paper Abstract :",df.iloc[idx]["abstract"][:500])
    print()

    summary = summarizer(df.iloc[idx]["abstract"],max_length=120, min_length=40,do_sample=False)
    print("Paper Summary :")
    print(summary[0]["summary_text"])
    print()

In [65]:
search_paper_and_summarize("Deep learning in medical imaging ",k=5)

Similarity Score : 0.7354985
Paper Title : A Perspective on Deep Imaging
Paper Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Paper Summary :
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity Score : 0.6881736
Paper Title : Classif

## Keyword Extraction using KeyBERT

In [66]:
from keybert import KeyBERT

In [67]:
kw_model = KeyBERT()

In [68]:
type(kw_model)

keybert._model.KeyBERT

In [72]:
text = df.iloc[10466]["abstract"]
keywords = kw_model.extract_keywords(text)
print(keywords)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [73]:
print(type(keywords))

<class 'list'>


In [74]:
print(type(keywords[0]))

<class 'tuple'>


**Extract Keywords using KeyBERT (n-gram based)**

In [75]:
keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1, 3),
    stop_words="english"
)
print(keywords)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


**Search, Summarize and Extract Keywords**

In [76]:
def search_summarize_extract_keywords(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40
        )

        print("Summary:")
        print(summary[0]["summary_text"])
        print()

        keywords = kw_model.extract_keywords(
            df.iloc[idx]["abstract"],
            keyphrase_ngram_range=(1, 3),
            stop_words="english"
        )

        print("Keywords:")

        for k_word, score in keywords:
            print(k_word)

        print("-" * 80)

In [77]:
search_summarize_extract_keywords("deep learning for medical image analysis", k=5)

Similarity score: 0.6807244
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Keywords:
tomographic imaging deep
imaging deep learning
learning medica

## Explainable Research Paper Recommendation System

**Research Paper Insight Generation**

In [78]:
def generate_insight(query, keywords, score):

    top_keywords = ", ".join(keywords[:5])

    # interpret similarity
    if score > 0.75:
        strength = "Very High"
        reason = "strong semantic match with your query"
    elif score > 0.60:
        strength = "High"
        reason = "good contextual similarity with your topic"
    else:
        strength = "Moderate"
        reason = "partial relevance to your query"

    return f"""
PAPER RELEVANCE ANALYSIS
----------------------------

Query Match Level : {strength}
Similarity Score   : {round(score, 4)}

Key Topics Identified:
{top_keywords}

Conclusion:
The paper demonstrates {reason} with respect to the input query.
It is therefore considered relevant for recommendation.
"""

**Test on Single Paper**

In [79]:
sample_index = 0

title = df.iloc[sample_index]["title"]
abstract = df.iloc[sample_index]["abstract"]

**Extract keywords**

In [80]:
keywords = kw_model.extract_keywords(
    abstract,
    keyphrase_ngram_range=(1, 3),
    stop_words="english"
)

keyword_list = [k for k, _ in keywords]

**Get similarity score**

In [81]:
query = "deep learning medical imaging"

query_embedding = model.encode([query])
paper_embedding = model.encode([abstract])

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

score = cosine_similarity(query_embedding, paper_embedding)[0][0]

**Generate Insight**

In [82]:
print("TITLE:", title)
print("\nKEYWORDS:", keyword_list)

print("\nINSIGHT:\n")
print(generate_insight(query, keyword_list, score))

TITLE: Learning from compressed observations

KEYWORDS: ['information theoretic characterization', 'characterization achievable predictor', 'function information theoretic', 'information theoretic', 'distribution allowable predictors']

INSIGHT:


PAPER RELEVANCE ANALYSIS
----------------------------

Query Match Level : Moderate
Similarity Score   : 0.1800999939441681

Key Topics Identified:
information theoretic characterization, characterization achievable predictor, function information theoretic, information theoretic, distribution allowable predictors

Conclusion:
The paper demonstrates partial relevance to your query with respect to the input query.
It is therefore considered relevant for recommendation.



In [83]:
def search_paper_with_insight(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        title = df.iloc[idx]["title"]
        abstract = df.iloc[idx]["abstract"]

        print("TITLE:", title)
        print("\nSIMILARITY SCORE:", score)
        print("\nABSTRACT:", abstract[:500])
        print()

        keywords = kw_model.extract_keywords(
            abstract,
            keyphrase_ngram_range=(1, 3),
            stop_words="english"
        )
        keyword_list = [k for k, _ in keywords]

        print("\nKEYWORDS:", keyword_list)

        print("\nWHY THIS PAPER:")
        print(generate_insight(query, keyword_list, score))

        print("-"*80)

In [84]:
search_paper_with_insight("deep learning for medical imaging")

TITLE: A Perspective on Deep Imaging

SIMILARITY SCORE: 0.7292017

ABSTRACT:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance


KEYWORDS: ['tomographic imaging deep', 'imaging deep learning', 'learning medical imaging', 'imaging deep', 'medical imaging']

WHY THIS PAPER:

PAPER RELEVANCE ANALYSIS
----------------------------

Query Match Level : High
Similarity Score   : 0.729200005531311

Key Topics Identified:
tomographic imaging deep, imaging deep learning, learning medical imaging, imaging deep, medical imaging

Conclusion:
The paper demo

## Research Paper Difficulty Analyzer

In [85]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.2 MB/s eta 0:00:00


In [86]:
import textstat
import re

In [87]:
def analyze_difficulty(text, keywords):

    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]

    avg_sentence_length = len(text.split()) / len(sentences) if len(sentences) > 0 else 0
    abstract_length = len(text.split())
    readability = textstat.flesch_reading_ease(text)
    keyword_count = len(keywords)

    sentence_score = min(avg_sentence_length / 40, 1)
    length_score = min(abstract_length / 300, 1)
    keyword_score = min(keyword_count / 15, 1)
    readability_score = max(0, (100 - readability) / 100)

    difficulty_score = (
        0.30 * sentence_score +
        0.30 * keyword_score +
        0.20 * length_score +
        0.20 * readability_score
    ) * 10

    if difficulty_score < 4:
        level = "Beginner"
    elif difficulty_score < 7:
        level = "Intermediate"
    else:
        level = "Advanced"

    return level, round(difficulty_score, 2)

In [88]:
def get_research_paper_insights(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        title = df.iloc[idx]["title"]
        abstract = df.iloc[idx]["abstract"]

        print("TITLE:", title)
        print("\nSIMILARITY SCORE:", score)
        print("\nABSTRACT:", abstract[:500])
        print()

        keywords = kw_model.extract_keywords(
            abstract,
            keyphrase_ngram_range=(1, 3),
            stop_words="english"
        )
        keyword_list = [k for k, _ in keywords]

        print("\nKEYWORDS:", keyword_list)

        print("\nWHY THIS PAPER:")
        print(generate_insight(query, keyword_list, score))

        level, score_diff = analyze_difficulty(abstract, keyword_list)

        print("\nDIFFICULTY ANALYSIS:")
        print("Level:", level)
        print("Score:", score_diff)
        print()
        print("-"*80)
        print()

In [89]:
get_research_paper_insights("deep learning for medical imaging")

TITLE: A Perspective on Deep Imaging

SIMILARITY SCORE: 0.7292017

ABSTRACT:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance


KEYWORDS: ['tomographic imaging deep', 'imaging deep learning', 'learning medical imaging', 'imaging deep', 'medical imaging']

WHY THIS PAPER:

PAPER RELEVANCE ANALYSIS
----------------------------

Query Match Level : High
Similarity Score   : 0.729200005531311

Key Topics Identified:
tomographic imaging deep, imaging deep learning, learning medical imaging, imaging deep, medical imaging

Conclusion:
The paper demo